# Predicting Bankruptcy of Polish Companies Using Gradient Boosting: Oversampling, Hyperparameter Tuning, Interactive Thresholding, and Model Saving for Deployment

1. Inspeksi struktur file

- Membuka file .json.gz dengan gzip dan json.load.
- Kunci tingkat atas: schema, data, metadata, yaitu tempat data disimpan.

In [ ]:
# Inspect the *structure* of a compressed JSON file (optional exploration).
import gzip
import json
from pathlib import Path

sample_path = Path("data/poland-bankruptcy-data-2009.json.gz")

with gzip.open(sample_path, "rt", encoding="utf-8") as f:
    payload = json.load(f)

# The top-level keys show how records are stored inside the file.
list(payload.keys())

2. Import

- RandomOverSampler untuk oversampling, GradientBoostingClassifier dan RandomForestClassifier sebagai model, SimpleImputer untuk nilai hilang.
- GridSearchCV untuk tuning, ipywidgets untuk slider interaktif, pickle untuk menyimpan model.
- from data import wrangle memakai fungsi wrangling dari pelajaran 1.

In [ ]:
# pip install pandas numpy matplotlib scikit-learn imbalanced-learn ipywidgets

In [ ]:
# Imports and display settings used throughout the notebook.
import pickle
from pathlib import Path

import ipywidgets as widgets
import numpy as np
import pandas as pd
from imblearn.over_sampling import RandomOverSampler
from ipywidgets import interact
from matplotlib import pyplot as plt
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import make_pipeline

from data import wrangle

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 120)

3. Memuat data dan menyiapkan fitur

- wrangle() menghasilkan DataFrame (9977, 66).
- feature_cols mengambil 64 kolom feat_* sebagai X, bankrupt (integer) sebagai y.
- assert memastikan ada 64 fitur dan target hanya berisi 0/1.

In [ ]:
# Point to the Poland dataset file and load it with wrangle().
poland_path = Path("data/poland-bankruptcy-data-2009.json.gz")
poland_df = wrangle(poland_path)

# Quick sanity-check: (rows, columns)
poland_df.shape

In [ ]:
# Checkpoint: ensure the supervised-learning target is present.
assert "bankrupt" in poland_df.columns, (
    "Expected 'bankrupt'. Make sure you loaded the full Poland dataset."
)

In [ ]:
# 1) Select the financial feature columns.
#    We keep only columns whose names start with "feat_" so we don't
#    accidentally include the target or any non-feature columns.
feature_cols = [c for c in poland_df.columns if c.startswith("feat_")]

# 2) Build X as a DataFrame with just the features.
#    X must be 2D (rows = firms, columns = financial indicators).
X = poland_df[feature_cols]

# 3) Build y as an integer Series (0/1).
#    y must be 1D and represent the label we want to predict.
#    Casting to int ensures labels are clean and consistent for metrics/models.
y = poland_df["bankrupt"].astype(int)

# Quick sanity check: X should have the same number of rows as y.
print(X.shape, y.shape)

In [ ]:
# Checkpoint: validate the dataset structure before training models.
# Poland is expected to have exactly 64 financial features (feat_1..feat_64).
assert X.shape[1] == 64, "Poland should have feat_1..feat_64"

# Checkpoint: ensure the target is strictly binary after casting.
assert set(y.unique()).issubset({0, 1})

4. Train/test split

- test_size=0.25, stratify=y, random_state=42.
- Rasio bangkrut train 4,68% vs test 4,69%, dan assert memastikan selisihnya kecil.

In [ ]:
# Split once and reuse these arrays for all models below.
# This ensures every model is trained and evaluated on the same data split.
X_train, X_test, y_train, y_test = train_test_split(
    X,             # feature matrix (2D)
    y,             # target labels (1D)
    test_size=0.25,  # keep 25% as a held-out test set
    random_state=42, # reproducible split across runs
    stratify=y,      # preserve class proportions in train and test
)

# Compare bankruptcy rates to confirm stratification worked.
# Because y is 0/1, the mean equals the fraction of bankrupt firms.
y_train.mean(), y_test.mean()

In [ ]:
# Checkpoint: train/test bankruptcy rates should be very close.
# If this fails, the split may not be stratified or something is off in y.
assert abs(y_train.mean() - y_test.mean()) < 0.01

5. Oversampling

- RandomOverSampler hanya pada data train: kelas menjadi seimbang 7.132 vs 7.132.
- Test set tidak disentuh agar evaluasi tetap realistis.

In [ ]:
# Oversample the minority class in the training set only.
# This increases the number of bankrupt examples by duplicating them
# until both classes have the same count.
ros = RandomOverSampler(
    random_state=42  # reproducible resampling
)

# Fit the oversampler on the training data and return the resampled arrays.
# IMPORTANT: do NOT resample X_test/y_test, or evaluation will be unrealistic.
X_train_over, y_train_over = ros.fit_resample(
    X_train,  # original training features
    y_train,  # original training labels
)

# After resampling, the class counts should be balanced (same number of 0s and 1s).
y_train_over.value_counts()

In [ ]:
# Checkpoint: oversampling should make both classes equally frequent.
counts = y_train_over.value_counts()
assert counts.iloc[0] == counts.iloc[1]

6. compute_metrics()

- Fungsi pembungkus yang menghitung precision, recall, F1 (dengan zero_division=0), dan ROC-AUC (memakai probabilitas, bukan label).

In [ ]:
def compute_metrics(
    y_true: pd.Series,
    y_pred: pd.Series,
    y_proba: pd.Series,
) -> dict[str, float]:
    """Compute core metrics for imbalanced binary classification.

    Parameters
    ----------
    y_true
        True labels (0/1).
    y_pred
        Predicted labels (0/1).
    y_proba
        Predicted probabilities for class 1.

    Returns
    -------
    dict[str, float]
        Precision, recall, F1, ROC–AUC.
    """
    # Precision: of the predicted bankruptcies, how many were correct?
    # zero_division=0 avoids errors when a model predicts zero positives.
    precision = precision_score(y_true, y_pred, zero_division=0)

    # Recall: of the true bankruptcies, how many did we detect?
    recall = recall_score(y_true, y_pred, zero_division=0)

    # F1: harmonic mean of precision and recall (useful under imbalance).
    f1 = f1_score(y_true, y_pred, zero_division=0)

    # ROC–AUC: evaluates how well predicted probabilities rank positives above negatives.
    # This uses y_proba (probability of class 1), not hard class labels.
    roc_auc = roc_auc_score(y_true, y_proba)

    # Return a consistent dictionary so all models can be compared uniformly.
    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
    }

7. Gradient Boosting baseline

- Pipeline SimpleImputer(median) lalu GradientBoostingClassifier, dilatih pada data oversampling.
- Hasil: precision 0,26, recall 0,65, F1 0,37, ROC-AUC 0,883.

In [ ]:
# Baseline Gradient Boosting (train on oversampled data).
# We use a pipeline so missing values are handled consistently via median imputation.
gb_baseline = make_pipeline(
    SimpleImputer(strategy="median"),  # fill missing numeric values
    GradientBoostingClassifier(random_state=42),  # boosting model (baseline)
)

# Fit on the oversampled training set so the model sees enough bankrupt examples.
gb_baseline.fit(X_train_over, y_train_over)

# Predict hard class labels (0/1) on the untouched test set.
y_pred_gb0 = gb_baseline.predict(X_test)

# Predict probabilities for the positive class (bankrupt = 1).
# We take [:, 1] because column 1 corresponds to class 1 probabilities.
y_proba_gb0 = gb_baseline.predict_proba(X_test)[:, 1]

# Compute the core metrics using the helper for consistent evaluation.
gb0_metrics = compute_metrics(y_test, y_pred_gb0, y_proba_gb0)
gb0_metrics

8. Random Forest (referensi)

- RF dengan class_weight="balanced", dilatih pada data asli.
- Hasil: precision 0,64, recall 0,06, F1 0,11, ROC-AUC 0,840.

In [ ]:
# Random Forest reference model (bagging).
# Here we train on the original (non-oversampled) training split and use
# class_weight="balanced" to compensate for imbalance during learning.
rf = make_pipeline(
    SimpleImputer(strategy="median"),  # same preprocessing as other models
    RandomForestClassifier(
        n_estimators=300,       # number of trees in the forest
        random_state=42,        # reproducibility
        n_jobs=-1,              # use all CPU cores
        class_weight="balanced" # weight classes inversely proportional to frequency
    ),
)

# Fit on the original training set (no oversampling here).
rf.fit(X_train, y_train)

# Predict hard labels on the test set.
y_pred_rf = rf.predict(X_test)

# Predict probabilities for class 1 (bankrupt).
y_proba_rf = pd.Series(rf.predict_proba(X_test)[:, 1])

# Score with the same helper to enable fair comparisons.
rf_metrics = compute_metrics(y_test, y_pred_rf, y_proba_rf)
rf_metrics

9. GridSearchCV untuk Gradient Boosting

- Grid: n_estimators [100, 200], learning_rate [0,05, 0,1], max_depth [2, 3], subsample [1,0, 0,8], dengan cv=3 dan scoring="roc_auc".
- Terbaik: n_estimators=200, learning_rate=0.1, max_depth=3, subsample=0.8; skor CV ROC-AUC 0,996.

In [ ]:
# Tuned Gradient Boosting using a small, readable grid.
# We keep the same preprocessing step to ensure tuning compares apples-to-apples.
gb_pipe = make_pipeline(
    SimpleImputer(strategy="median"),  # fill missing numeric values
    GradientBoostingClassifier(random_state=42),  # boosting model (baseline)
)

# Grid over a few key hyperparameters:
# - n_estimators: number of boosting stages => [100, 200]
# - learning_rate: step size shrinkage => [0.05, 0.1]
# - max_depth: complexity of each base learner (via tree depth) => [2, 3]
# - subsample: stochastic boosting when < 1.0 => [1.0, 0.8]
param_grid = {
    "gradientboostingclassifier__n_estimators": [100, 200],
    "gradientboostingclassifier__learning_rate": [0.05, 0.1],
    "gradientboostingclassifier__max_depth": [2, 3],
    "gradientboostingclassifier__subsample": [1.0, 0.8],
}

# Use ROC–AUC for selection because it evaluates probability ranking
# and is commonly used for imbalanced binary classification.
grid = GridSearchCV(
    gb_pipe,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=3,     # small CV for speed while still validating generalization
    n_jobs=-1 # parallelize across CPU cores
)

# Fit the search on the oversampled training data (same as baseline GB training).
grid.fit(X_train_over, y_train_over)

# Inspect the best hyperparameters and best cross-validated ROC–AUC.
grid.best_params_, grid.best_score_

10. Evaluasi model tuned dan perbandingan

In [ ]:
# Evaluate the best tuned model on the test set.
# best_estimator_ is the full pipeline (imputer + tuned gradient boosting).
gb_best = grid.best_estimator_

# Predict test labels and probabilities using the tuned pipeline.
y_pred_gb = gb_best.predict(X_test)
y_proba_gb = pd.Series(gb_best.predict_proba(X_test)[:, 1])

# Compute test metrics for the tuned model.
gb_metrics = compute_metrics(y_test, y_pred_gb, y_proba_gb)
gb_metrics

In [ ]:
# Combine metrics into a single comparison table.
# Each list item becomes one row in the final DataFrame.
# We store a readable model name under "model" and then unpack the metric dict.
comparison = pd.DataFrame(
    [
        {"model": "Random Forest", **rf_metrics},         # bagging reference
        {"model": "GradBoost (baseline)", **gb0_metrics},  # boosting baseline
        {"model": "GradBoost (tuned)", **gb_metrics},      # tuned boosting model
    ]
)

# Use the model name as the row index so the table reads naturally.
comparison = comparison.set_index("model")

# Display the comparison table (rows = models, columns = metrics).
comparison

In [ ]:
# Checkpoint: confirm that all expected metric columns are present.
# This helps catch missing keys or earlier metric computation issues.
assert {"precision", "recall", "f1", "roc_auc"}.issubset(comparison.columns)

11. Confusion matrix dan classification report

- Dari 117 perusahaan bangkrut di test, sekitar 58% terdeteksi. Akurasi 93,6%, tetapi kelas 1 hanya memiliki precision 0,382 dan F1 0,461.

In [ ]:
# Confusion matrix + report for the tuned Gradient Boosting model.
# The confusion matrix shows counts of TN / FP / FN / TP on the test set.
ConfusionMatrixDisplay.from_predictions(
    y_test,        # true labels
    y_pred_gb,     # predicted labels from the tuned model at the default threshold
    values_format="d",  # show integer counts
)

# Add a descriptive title so the plot is self-explanatory in the notebook.
plt.title("Tuned Gradient Boosting — Confusion Matrix (Test)")
plt.show()

# The classification report gives precision/recall/F1 for each class (0 and 1).
# digits=3 improves readability; zero_division=0 avoids warnings if a class
# has no predicted positives.
print(classification_report(y_test, y_pred_gb, digits=3, zero_division=0))

12. Threshold interaktif

- threshold_demo() mengubah probabilitas menjadi label dengan cutoff tertentu, lalu menampilkan metrik dan confusion matrix.
- Slider 0,05-0,95; pada threshold 0,25 recall naik ke 0,78 tetapi precision turun ke 0,19.

In [ ]:
# Optional: explore thresholds to see the precision/recall trade-off.
def threshold_demo(threshold: float = 0.5) -> None:
    """Display metrics and a confusion matrix for a chosen threshold."""
    # Convert probabilities into class predictions using the chosen cutoff.
    # If proba >= threshold => predict bankruptcy (1), else non-bankrupt (0).
    y_pred_thr = (y_proba_gb >= threshold).astype(int)

    # Compute the confusion matrix counts for this threshold.
    cm = confusion_matrix(
        y_test,     # true labels
        y_pred_thr, # thresholded predictions
    )

    # Print key metrics so you can see how the threshold shifts the trade-off.
    print(
        "precision:",
        precision_score(y_test, y_pred_thr, zero_division=0),
        "recall:",
        recall_score(y_test, y_pred_thr, zero_division=0),
        "f1:",
        f1_score(y_test, y_pred_thr, zero_division=0),
    )

    # Plot the confusion matrix for the chosen threshold.
    ConfusionMatrixDisplay(cm).plot(values_format="d")
    plt.title(f"Threshold = {threshold:.2f}")
    plt.show()

13. Simpan dan muat model

- Pipeline terbaik disimpan ke models/poland_gradient_boosting.pkl dengan pickle, lalu dimuat kembali.
- Metrik model termuat sama persis, dan assert np.allclose memastikan probabilitasnya identik.

In [ ]:
# Try a few thresholds and observe how the confusion matrix changes.
# Use a slider to interactively choose the cutoff and rerun threshold_demo.
interact(
    threshold_demo,
    threshold=widgets.FloatSlider(
        value=0.5,  # default scikit-learn threshold
        min=0.05,   # lower cutoff to prioritize recall
        max=0.95,   # higher cutoff to prioritize precision
        step=0.05,  # coarse steps for quick exploration
    ),
);

In [ ]:
# Create a folder to store trained models.
model_dir = Path("models")
model_dir.mkdir(exist_ok=True)

# Save the tuned Gradient Boosting model to .
model_path = model_dir / "poland_gradient_boosting.pkl"

with model_path.open("wb") as f:
    pickle.dump(gb_best, f)

model_path

In [ ]:
# Load the model back from disk.
with model_path.open("rb") as f:
    loaded_model = pickle.load(f)

In [ ]:
# Predict again on X_test to confirm the loaded model works.
loaded_proba = loaded_model.predict_proba(X_test)[:, 1]
loaded_pred = (loaded_proba >= 0.5).astype(int)

loaded_metrics = compute_metrics(
    y_test,
    pd.Series(loaded_pred),
    pd.Series(loaded_proba),
)
loaded_metrics

In [ ]:
# Checkpoint: loaded model should reproduce the same probabilities.
assert np.allclose(loaded_proba, y_proba_gb.values)

14. predictor.py

- Menulis modul dengan fungsi predict_bankruptcy_prob(model_path, data_path): memuat model, memanggil wrangle, mengambil kolom feat_*, dan mengembalikan probabilitas bangkrut sebagai pd.Series.
- Blok __main__ memungkinkan dijalankan langsung sebagai skrip.

In [ ]:
predictor_code = '''
from __future__ import annotations

import pickle
from pathlib import Path

import pandas as pd

from data import wrangle

def predict_bankruptcy_prob(
    model_path: str | Path,
    data_path: str | Path,
) -> pd.Series:
    """Predict bankruptcy probabilities for a dataset file.

    Parameters
    ----------
    model_path
        Path to a trained scikit-learn pipeline saved with pickle.
    data_path
        Path to a dataset file compatible with ``wrangle``.

    Returns
    -------
    pd.Series
        Predicted probabilities for class 1 (bankrupt), indexed by row id.
    """
    model_path = Path(model_path)
    data_path = Path(data_path)

    with model_path.open("rb") as f:
        model = pickle.load(f)

    df = wrangle(data_path)
    feat_cols = [c for c in df.columns if c.startswith("feat_")]
    X = df[feat_cols]

    proba = model.predict_proba(X)[:, 1]
    return pd.Series(proba, index=df.index)

if __name__ == "__main__":
    model_path = Path("models/poland_gradient_boosting.pkl")
    data_path = Path("data/poland-bankruptcy-data-2009.json.gz")

    probs = predict_bankruptcy_prob(model_path, data_path)
    print(probs.head())
'''

Path("predictor.py").write_text(predictor_code, encoding="utf-8")
Path("predictor.py")

Kesimpulan Singkat

- Gradient Boosting jauh lebih baik dalam mendeteksi kebangkrutan. 
- Recall naik dari 0,06 (Random Forest) menjadi 0,58, dan ROC-AUC naik dari 0,840 menjadi 0,894.
- Oversampling dan tuning efektif. 
- Tuning menaikkan precision (0,26 menjadi 0,38) dan F1 (0,37 menjadi 0,46) dengan recall tetap tinggi.
- Ada trade-off precision-recall. 
- Model tuned menangkap 58% perusahaan bangkrut, tetapi sekitar 62% peringatannya salah (false alarm). 
- Threshold dapat disesuaikan sesuai kebutuhan bisnis: recall tinggi untuk deteksi dini, precision tinggi untuk mengurangi alarm palsu.
- Skor CV (0,996) tidak realistis. 
- Angka itu jauh di atas ROC-AUC test (0,894) karena oversampling dilakukan sebelum cross-validation, sehingga duplikat sampel bocor antar fold. 
- Gunakan pipeline imblearn agar oversampling hanya terjadi di dalam tiap fold.
- Proyek sudah siap deployment: model tersimpan sebagai .pkl dan predictor.py memberi fungsi prediksi yang reusable.

Saran lanjutan: perbaiki kebocoran CV, pilih threshold berdasarkan kurva Precision-Recall, coba SMOTE atau XGBoost/LightGBM, dan tambahkan validasi input pada predictor.py.